## DERIVATIVE PRICING
MODULE 2 | LESSON 4


---



# **INTRO TO MONTE CARLO METHODS**


|  |  |
|:---|:---|
|**Reading Time** |  60 minutes |
|**Prior Knowledge** | Binomial model, Option pricing, European options, American options, Asian options |
|**Keywords** |Monte-Carlo methods, Path-dependent options |


---<span style='color: transparent; font-size:1%'>All rights reserved WQU WorldQuant University QQQQ</span>

*In Lesson 4 of Module 2, we will go through the basics of using Monte Carlo methods for derivative pricing. Ideally, you should look at this notebook while working with the corresponding video and slides, but you can also work on this on your own.*


## **1. Monte Carlo Methods**

Please note that we are introducing Monte Carlo methods informally here. Monte Carlo comes in handy in a lot of derivative pricing problems in real life, and we will cover this method thoroughly throughout the course. We believe this simple introduction will help down the road regarding basic intuition. The mapping of what the actual analytical computation does versus the Monte Carlo estimation will be very easy to see in this simple setting. In the future, it can be more complex though, so we believe it is best to introduce it here.

Let's start importing the numpy library:

In [1]:
import numpy as np

Next, we are going to develop a function, `call_option_mc`, that will use Monte Carlo-like methods to study the power of this technique.

The basic intuition of Monte Carlo is that by simulating a fair amount of outcomes randomly from the same distribution as our underlying process, we can obtain a solution close to the actual (full) process. The process will go as follows:
- Select the distribution you are using for the underlying process: In our case, it is a binomial distribution for the stock price. 
- Simulate random realizations of such distribution: We use Python to choose a random number from a binomial distribution using np.random.binomial.
- Calculate option payoff for those simulated values and discount them so that you can confirm a price for the option under each simulated path.
- Get the mean of those prices: By relying on the law of large numbers, when the amount of randomly generated values is high enough, the average of all converges to the true value of the options. Of course, the more simulations, the merrier.

Let's get to it with the European call option example:

## **2. European Option Payoff (Monte Carlo)**

In [2]:
def call_option_mc(S_ini, K, T, r, sigma, N, M):
    dt = T / N  # Define time step
    u = np.exp(sigma * np.sqrt(dt))  # Define u
    d = np.exp(-sigma * np.sqrt(dt))  # Define d
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([M])  # call prices
    S = np.zeros([M, N + 1])  # underlying price
    S[:, 0] = S_ini

    for j in range(0, M):
        random = np.random.binomial(
            1, p, N + 1
        )  # We sample random realizations for the paths of the tree under a binomial distribution
        for i in range(1, N + 1):
            if random[i] == 1:
                S[j, i] = S[j, i - 1] * u
            else:
                S[j, i] = S[j, i - 1] * d

        C[j] = np.exp(-r * T) * max(S[j, N] - K, 0)

    return S, C

Now that we have our function, let's check its performance by comparing it to the European call option we priced in Module 1. If you remember, these were the characteristics of the option:
- $S_0=100$
- $K=90$
- $T=1$
- $r = 0\%$
- $\sigma = 0.3$

If you remember, when we increase the number of paths in the binomial tree, $N$, the price of the option converged to $\$17.01$ for $N=2,500$.

So, let's see how our (quasi) Monte Carlo algorithm performs for $N=2,500$ using $15,000$ simulations (paths):

In [3]:
S, C = call_option_mc(
    100, 90, 1, 0, 0.3, 2500, 15000
)  # Prices 15000 different simulations

Now the variable $C$ contains the $15,000$ different prices from the different simulations performed. In order to get the final value for the call option proxied with Monte Carlo, we just average all these values:

In [4]:
print(np.mean(C))

16.618959414904065


As you can see, the value is pretty close to the one we got from the workout on the full binomial model of $\$17.01$.

But let's dig a little deeper into the convergence to this value. We will now check how we start approximating to this value as we increase the number of simulations:

In [5]:
M = np.arange(1000, 16000, 1000)  # Different number of simulations to be considered
call_price = []

*Warning: The following code will take several minutes to run!*

In [ ]:
for i in range(len(M)):
    S, C = call_option_mc(100, 90, 1, 0, 0.3, 2500, M[i])
    call_price.append(np.mean(C))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(M, call_price)
plt.axhline(y=17.01, color="r", linestyle="-", label="Binomial model price")
plt.ylim([14, 23])
plt.title("MC Estimates for different # simulations")
plt.xlabel("Number of Simulations")
plt.ylabel("Call Option Price")
plt.grid(True)
plt.legend()
plt.show()

While the past exercise may seem in vain, given that we can simply use the binomial model to price the option, it gives us a pretty good idea of how the Monte Carlo method works.

Understanding Monte Carlo methods, thus, will help us provide efficient solutions/pricing in more problematic scenarios, as was the case with the Asian option. In this framework, calculating ALL the potential payoffs of the option for $N=2,500$ would mean we have to deal with $2^N = 2^{2500}$ paths. That is virtually impossible and inefficient. 

Hence, Monte Carlo will come in handy in these type of situations for path-dependent derivatives. Let's now build up a function, `asian_option_mc`, to price an Asian option with the same characteristics using this technique:

## **3. Asian Option Payoff (Monte Carlo)**

In [ ]:
def asian_option_mc(S_ini, K, T, r, sigma, N, M):
    dt = T / N  # Define time step
    u = np.exp(sigma * np.sqrt(dt))  # Define u
    d = np.exp(-sigma * np.sqrt(dt))  # Define d
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    Asian = np.zeros([M])  # Asian prices
    S = np.zeros([M, N + 1])  # underlying price
    S[:, 0] = S_ini

    for j in range(0, M):
        random = np.random.binomial(1, p, N + 1)
        Total = S_ini
        for i in range(1, N + 1):
            if random[i] == 1:
                S[j, i] = S[j, i - 1] * u
                Total = Total + S[j, i]
            else:
                S[j, i] = S[j, i - 1] * d
                Total = Total + S[j, i]

        Asian[j] = np.exp(-r * T) * max(Total / (N + 1) - K, 0)

    return S, Asian

As before, let's check the prices for the different paths:

In [ ]:
S, Asian = asian_option_mc(100, 90, 1, 0, 0.3, 2500, 10000)

And average them to get a final estimate:

In [ ]:
print(np.mean(Asian))

Let's also study the convergence of the methods in a similar fashion as before:

In [ ]:
M = np.arange(1000, 16000, 1000)
asian_price = []

*Warning: The following code will take several minutes to run!*

In [ ]:
for i in range(len(M)):
    S, Asian = asian_option_mc(100, 90, 1, 0, 0.3, 2500, M[i])
    asian_price.append(np.mean(Asian))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(M, asian_price)
plt.ylim([10, 15])
plt.title("MC Estimates for different # simulations")
plt.xlabel("Number of Simulations")
plt.ylabel("Asian Call Option Price")
plt.grid(True)
plt.show()

## **4. Conclusion**

All in all, you have seen firsthand how mastering Monte Carlo methods can be a fundamental ally when it comes to pricing complex derivatives for which it is too costly (or impossible) to actually compute all the potential outcomes of the underlying process.

Please note once again that this lesson introduces Monte Carlo from an informal perspective and only with the goal of simplifying your understanding of it. We will dig deeper into this in future modules.

---
Copyright 2025 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
